# FAISS (Facebook AI Similarity Search)

In [22]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [23]:
import torch
from plantclef.config import get_device

print(f"PyTorch Version: {torch.__version__}")
device = get_device()
print(f"Using device: {device}")

PyTorch Version: 2.6.0+cu124
Using device: cuda


In [24]:
import pandas as pd
from pathlib import Path

# Get list of stored filed in cloud bucket
root = Path().resolve().parents[0]
print(root)
! date

/storage/home/hcoda1/9/mgustineli3/clef/pytorch-plantclef
Mon Mar 24 11:58:36 AM EDT 2025


In [25]:
# path to data
data_path = f"{root}/data/embeddings"
train_path = f"{data_path}/train_embeddings"
test_path = f"{data_path}/test_grid_3x3_embeddings"

# read train/test data
train_df = pd.read_parquet(train_path)
test_df = pd.read_parquet(test_path)

# display data
print(f"Train DF shape: {train_df.shape}")
print(f"Test DF shape: {test_df.shape}")
display(train_df.head(3))
display(test_df.head(3))

Train DF shape: (2020, 8)
Test DF shape: (1800, 6)


,image_name,data,species,species_id,embeddings,logits,tile,partition
0,28e2fbd0cc93d82d7de3ed5783c64816074955e0.jpg,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00...,Pteridium aquilinum (L.) Kuhn,1356404,"[0.22075553238391876, -0.5152621269226074, 0.7...","{""1389294"": 0.4966914653778076, ""1356390"": 0.0...",0,0
1,47b5db375e741fdc09b5e133d5bfef0285aa695c.jpg,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00...,Carex firma Host,1418612,"[0.55893474817276, 0.19274836778640747, 0.8937...","{""1418612"": 0.7230438590049744, ""1390910"": 0.0...",0,0
2,c6a6b172ab03374ad1bb713b7e7b37dd84f094a1.jpg,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00...,Daucus pumilus (L.) Hoffmanns. & Link,1722578,"[-0.15363329648971558, 0.35160958766937256, 0....","{""1722578"": 0.8078303337097168, ""1411700"": 0.0...",0,0


,image_name,data,embeddings,logits,tile,partition
0,CBN-PdlC-C4-20180723.jpg,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x01...,"[-1.3391107320785522, 1.6340396404266357, -2.2...","{""1741880"": 0.1965922713279724, ""1729043"": 0.0...",0,0
1,CBN-PdlC-C4-20180723.jpg,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x01...,"[-0.11381672322750092, 1.790043830871582, -0.5...","{""1395807"": 0.19840383529663086, ""1741880"": 0....",1,0
2,CBN-PdlC-C4-20180723.jpg,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x01...,"[-0.40855133533477783, 1.9910684823989868, -0....","{""1395807"": 0.4447108805179596, ""1397468"": 0.0...",2,0


In [34]:
import torch
import faiss
import numpy as np
from plantclef.config import get_device


class FaissClassifier:
    def __init__(self, train_df: pd.DataFrame):
        """
        :param train_df: DataFrame with columns ["species_id", "embeddings"]
        """
        self.device = get_device()
        self.index, self.idx2cls = self.build_index(train_df)

    def build_index(self, train_df):
        """Builds the FAISS index from the training data."""

        # store class labels
        idx2cls = train_df["species_id"].values
        # convert embeddings to tensor
        embs_array = np.array(train_df["embeddings"].tolist(), dtype=np.float32)
        embs = torch.tensor(embs_array, device=self.device)
        # normalize embeddings for cosine similarity
        embs = torch.nn.functional.normalize(embs, p=2, dim=1)
        # create FAISS index
        index = faiss.IndexFlatIP(embs.shape[1])  # inner product (dot product)
        index.add(embs.cpu().numpy())  # FAISS expects numpy arrays
        return index, idx2cls

    def make_prediction(self, query_embeddings: torch.Tensor, k=1):
        """
        Predicts the class of given embeddings using nearest neighbor search.
        :param query_embeddings: tensor of shape (N, D) where N is the number of embeddings and D is the embedding dimension
        :param k: number of nearest neighbors to return
        :return: predictions, similarities
        """

        # normalize embeddings for cosine similarity
        query_embeddings = torch.nn.functional.normalize(query_embeddings, p=2, dim=1)
        # perform search
        similarities, indices = self.index.search(query_embeddings.cpu().numpy(), k=k)
        predictions = self.idx2cls[indices]
        return predictions, similarities

## similarity search using FAISS

In [42]:
# similarity seearch using FAISS
nn_classifier = FaissClassifier(train_df)

# convert test embeddings to torch tensor
embs_array = np.array(test_df["embeddings"].tolist(), dtype=np.float32)
query_embs = torch.tensor(embs_array, device=get_device())
cls, conf = nn_classifier.make_prediction(query_embs, k=1)
cls.shape, conf.shape

((1800, 1), (1800, 1))

In [43]:
cls[:10], conf[:10]

(array([[1393659],
        [1741880],
        [1395807],
        [1396094],
        [1396869],
        [1697384],
        [1397070],
        [1396869],
        [1393660],
        [1363945]], dtype=int32),
 array([[0.40008646],
        [0.46044004],
        [0.4139103 ],
        [0.39205796],
        [0.45508015],
        [0.43824112],
        [0.40617254],
        [0.45850965],
        [0.3880484 ],
        [0.29808292]], dtype=float32))

In [44]:
# top 5 predictions for each embedding
cls, conf = nn_classifier.make_prediction(query_embs, k=5)
cls[:10], conf[:10]

(array([[1393659, 1396869, 1741880, 1741880, 1741880],
        [1741880, 1360591, 1395807, 1741880, 1395807],
        [1395807, 1741880, 1394540, 1395974, 1395807],
        [1396094, 1397070, 1397070, 1391313, 1397070],
        [1396869, 1390899, 1741880, 1397070, 1390899],
        [1697384, 1395974, 1697384, 1397070, 1741880],
        [1397070, 1396094, 1396869, 1741880, 1397070],
        [1396869, 1741880, 1395807, 1741880, 1741880],
        [1393660, 1393659, 1393660, 1396869, 1741880],
        [1363945, 1363945, 1363945, 1363945, 1363945]], dtype=int32),
 array([[0.40008646, 0.34887767, 0.33780566, 0.33685827, 0.32239807],
        [0.46044004, 0.42304528, 0.42006686, 0.40647435, 0.40348232],
        [0.4139103 , 0.41236427, 0.41210604, 0.40539494, 0.4041413 ],
        [0.39205796, 0.39106837, 0.38321644, 0.3758384 , 0.35727823],
        [0.45508015, 0.4180646 , 0.40695882, 0.40578797, 0.4007255 ],
        [0.43824112, 0.41201875, 0.40011925, 0.3937057 , 0.39266932],
        [0.4061

In [45]:
cls.shape, conf.shape

((1800, 5), (1800, 5))